# Comparing multiple BDT algorithms with TMVA

In this example we will still do Higgs classification but we will compare several algorithms with the Toolkit for Multivariate Data Analysis with ROOT (TMVA) to classify events from particle collisions and quantify algorithms performance.

In [ ]:
!apt-get update
!apt-get install -y g++ make cmake git libx11-dev libxpm-dev libxft-dev libxext-dev
!apt-get install libpython3.12-dev

In [ ]:
# Install ROOT
!wget -O root_v6.32.00.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz https://root.cern/download/root_v6.32.00.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz
!tar -xzf root_v6.32.00.Linux-ubuntu22.04-x86_64-gcc11.4.tar.gz
!mv root root-install
!echo 'source /content/root-install/bin/thisroot.sh' >> ~/.bashrc
!source /content/root-install/bin/thisroot.sh

In [ ]:
import sys
import os

os.environ["ROOTSYS"] = "/content/root-install"
os.environ["PATH"] = "/content/root-install/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/content/root-install/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PYTHONPATH"] = "/content/root-install/lib:" + os.environ.get("PYTHONPATH", "")
sys.path.append("/content/root-install/lib")
#!python3 --version

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Declare Factory

Create the Factory class. Later you can choose the methods
whose performance you'd like to investigate.

The factory is the major TMVA object you have to interact with. Here is the list of parameters you need to pass

 - The first argument is the base of the name of all the output
weightfiles in the directory weight/ that will be created with the
method parameters

 - The second argument is the output file for the training results
  
 - The third argument is a string option defining some general configuration for the TMVA session. For example all TMVA output can be suppressed by removing the "!" (not) in front of the "Silent" argument in the option string

In [ ]:
# Importing the ROOT module
import ROOT
# Importing the TMVA, TFile (to open ROOT files) and TCut (to filter data)
from ROOT import TMVA, TFile, TCut

# Initialising TMVA
TMVA.Tools.Instance()
#TMVA.PyMethodBase.PyInitialize()

# This will overwrite the file at each execution
outputFile = TFile( "TMVA-Classification-Higgs-BDT-Comparison.root", "RECREATE" )
factory = TMVA.Factory( "TMVAClassification", outputFile,
                        "!V:Color:DrawProgressBar:Transformations=I;D;P;G,D:AnalysisType=Classification" )

## Input Data

We define now the input data file and we retrieve the ROOT TTree objects with the signal and background input events

In [ ]:
# Opening the input data
inputFileName = "/content/drive/MyDrive/Higgs_data.root"
inputFile = TFile.Open( inputFileName )

# Retrieve input trees
sigTree = inputFile.Get("sig_tree")
bkgTree = inputFile.Get("bkg_tree")

## Declare DataLoader(s)

The next step is to declare the DataLoader class that deals with input data abd variables

We add first the signal and background trees in the data loader and then we
define the input variables that shall be used for the MVA training
note that you may also use variable expressions, which can be parsed by TTree::Draw( "expression" )]

In [ ]:
#Instantiating the DataLoader
loader = TMVA.DataLoader("dataset")

# Global event weights per tree: useful to adjust the event weight to match a given luminosity
sigWeight = 1.0
bkgWeight = 1.0

# Note: one can add any number of trees (e.g. multiple backgrounds)
loader.AddSignalTree    ( sigTree, sigWeight )
loader.AddBackgroundTree( bkgTree, bkgWeight )

# Adding variables (here corresponding to the branches of the TTree)
loader.AddVariable("m_jj")
loader.AddVariable("m_jjj")
loader.AddVariable("m_lv")
loader.AddVariable("m_jlv")
loader.AddVariable("m_bb")
loader.AddVariable("m_wbb")
loader.AddVariable("m_wwbb")

## Setup Dataset(s)

Setup the DataLoader by splitting events in training and test samples.
Here we use a random split and a fixed number of training and test events.

In [ ]:
# Apply additional cuts on the signal and background samples (can be different)
sigCut = TCut("")   ## for example: TCut sigCut = "m_jj>100"
bkgCut = TCut("")   ## for example: TCut bkgCut = "m_jj>100"


loader.PrepareTrainingAndTestTree( sigCut, bkgCut,
         "nTrain_Signal=7000:nTrain_Background=7000:SplitMode=Random:"
         "NormMode=NumEvents:!V" )

# Booking Methods

Here we book the TMVA methods. We book a Likelihood based a BDT and a standard MLP (shallow NN)

In [ ]:
# Booking the BDT
factory.BookMethod(loader,ROOT.TMVA.Types.kBDT, "BDT",
          "!V:NTrees=200:MinNodeSize=2.5%:MaxDepth=2:BoostType=AdaBoost:AdaBoostBeta=0.5:UseBaggedBoost:"
          "BaggedSampleFraction=0.5:SeparationType=GiniIndex:nCuts=20" )


# Booking another BDT. Note that the methog has a different name
factory.BookMethod(loader,ROOT.TMVA.Types.kBDT, "BDT2",
          "!V:NTrees=100:MinNodeSize=2.5%:MaxDepth=2:BoostType=AdaBoost:AdaBoostBeta=0.5:UseBaggedBoost:"
          "BaggedSampleFraction=0.5:SeparationType=GiniIndex:nCuts=20" )

# Yet another BDT
factory.BookMethod(loader,ROOT.TMVA.Types.kBDT, "BDT3",
          "!V:NTrees=400:MinNodeSize=2.5%:MaxDepth=2:BoostType=AdaBoost:AdaBoostBeta=0.5:UseBaggedBoost:"
          "BaggedSampleFraction=0.5:SeparationType=GiniIndex:nCuts=20" )

## Train Methods

In [ ]:
factory.TrainAllMethods()

## Test  all methods

Here we test all methods using the test data set

In [ ]:
factory.TestAllMethods()

## Evaluate all methods

Here we evaluate all methods and compare their performances, computing efficiencies, ROC curves etc.. using both training and tetsing data sets. Several histograms are produced which can be examined with the TMVAGui or directly using the output file

In [ ]:
factory.EvaluateAllMethods()

## Plot ROC Curve
We enable JavaScript visualisation for the plots

In [ ]:
# We can now compare with our first BDT
cROC = factory.GetROCCurve(loader)
cROC.Draw()

In [ ]:
# Closing the file where we have the methods trained
outputFile.Close()
factory.Delete()

## Follow-up

You can test various BDT configurations to understand how these algorithms work.

### Exploring the methods using the TMVAGui

TMVA comes with the TMVAGui, which helps understand how the algorithms behave. To run it, simply run
```
root -l -e 'TMVA::TMVAGui("TMVA-Classification-Higgs-BDT-Comparison.root")'
```
from your terminal.